# Portfolio Project: SpaceX Falcon 9 First-Stage Landing Prediction
## 02 — Web Scraping Historical Launch Records

**Portfolio Project | Business Analytics & Data Analytics**

This notebook develops a second **data-acquisition pipeline** for the **SpaceX portfolio** by extracting historical launch records from a fixed **Wikipedia** revision (scrapped from Wiki page titled `List of Falcon 9 and Falcon Heavy launches`). While Notebook 01 demonstrates API-based acquisition and entity enrichment, this notebook focuses on **semi-structured web data**: locating HTML tables, cleaning embedded annotations, standardizing text fields, validating the extracted schema, and exporting a structured dataset.

Source Page: https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches

### Research role
Public web pages often contain operational information that is analytically useful but not distributed as tidy datasets. This notebook therefore demonstrates a reproducible web-scraping workflow that converts historical launch tables into a tabular form suitable for downstream analysis.

> **Historical snapshot note.** The scraper targets Wikipedia revision `1027686922`, dated **9 June 2021**. Using a fixed revision reduces source drift relative to scraping the continuously updated live article.

![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/Falcon9_rocket_family.svg)


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/falcon9-launches-wiki.png)


## 1. Environment and configuration

The notebook can operate in two modes:

- `RUN_LIVE_SCRAPE = True`: download and parse the fixed Wikipedia revision.
- `RUN_LIVE_SCRAPE = False`: load the archived `spacex_web_scraped.csv` produced by the original successful scraping run.

The archived mode is the default in this portfolio copy so that the notebook can be executed in environments without outbound internet access. The full refactored scraper remains included and can be activated with one configuration change.

In [1]:
from pathlib import Path
import re
import unicodedata

import pandas as pd
import requests
from bs4 import BeautifulSoup

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

STATIC_URL = (
    "https://en.wikipedia.org/w/index.php?"
    "title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"
)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/91.0.4472.124 Safari/537.36"
    )
}

ARCHIVED_OUTPUT = Path("spacex_web_scraped.csv")
EXPORT_PATH = Path("spacex_web_scraped_portfolio.csv")

# Set to True to rerun the scraper against the fixed historical Wikipedia revision.
RUN_LIVE_SCRAPE = False

print(f"Live scrape enabled: {RUN_LIVE_SCRAPE}")

Live scrape enabled: False


## 2. Parsing utilities

Wikipedia launch tables contain nested links, footnotes, line breaks, and inconsistent text fragments. The following helper functions isolate the fields needed for analysis while avoiding assumptions that every cell contains the same HTML structure.

In [2]:
def clean_text(cell):
    """Return normalized visible text from a BeautifulSoup cell."""
    if cell is None:
        return None
    text = unicodedata.normalize("NFKD", cell.get_text(" ", strip=True))
    text = re.sub(r"\[[^\]]*\]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text or None


def first_link_text(cell):
    """Return the first link text when available; otherwise return cleaned cell text."""
    if cell is None:
        return None
    link = cell.find("a")
    return clean_text(link) if link else clean_text(cell)


def parse_date_time(cell):
    """Extract date and UTC time from the combined date/time table cell."""
    pieces = [clean_text_piece for clean_text_piece in cell.stripped_strings]
    pieces = [p.strip() for p in pieces if p.strip()]
    date = pieces[0].rstrip(",") if pieces else None
    time = pieces[1] if len(pieces) > 1 else None
    return date, time


def parse_booster_version(cell):
    """Extract a compact booster-version label from the version/booster cell."""
    if cell is None:
        return None

    # Prefer the visible text, then remove footnote markers and excess whitespace.
    value = clean_text(cell)
    if value:
        value = re.sub(r"\s+", "", value)
    return value


def parse_payload_mass(cell):
    """Preserve the historical payload-mass representation in kilograms when available."""
    if cell is None:
        return "0"

    text = unicodedata.normalize("NFKD", cell.get_text(" ", strip=True))
    match = re.search(r"([\d,]+(?:\.\d+)?)\s*kg", text, flags=re.IGNORECASE)

    if not match:
        return "0"

    return f"{match.group(1)} kg"


def parse_landing_status(cell):
    """Extract booster-landing status as normalized text."""
    return clean_text(cell)


def extract_column_from_header(header):
    """Clean a table-header cell without mutating the original BeautifulSoup tree."""
    clone = BeautifulSoup(str(header), "html.parser")

    for tag in clone.find_all(["br", "sup"]):
        tag.decompose()

    text = clean_text(clone)
    if not text or text.isdigit():
        return None

    return text

## 3. Source retrieval

The request logic is isolated in a function with an explicit timeout and `raise_for_status()`. This is more robust than allowing unsuccessful HTTP responses to propagate silently into the parser.

In [3]:
def fetch_wikipedia_snapshot(url=STATIC_URL, headers=HEADERS, timeout=30):
    """Download the fixed historical Wikipedia page and return a parsed soup object."""
    response = requests.get(url, headers=headers, timeout=timeout)
    response.raise_for_status()

    soup = BeautifulSoup(response.content, "html.parser")

    if "Falcon 9" not in soup.get_text(" ", strip=True):
        raise ValueError("Downloaded page does not appear to contain the expected Falcon 9 launch history.")

    return soup

## 4. Identify launch tables by schema

The notebook identifies launch tables by checking whether their header text contains the expected launch-record fields.

In [4]:
EXPECTED_HEADER_TERMS = {
    "Flight No.",
    "Launch site",
    "Payload",
    "Payload mass",
    "Orbit",
    "Customer",
}


def table_header_text(table):
    headers = [
        extract_column_from_header(th)
        for th in table.find_all("th")
    ]
    return {h for h in headers if h}


def is_launch_table(table):
    headers = table_header_text(table)
    return EXPECTED_HEADER_TERMS.issubset(headers)


def find_launch_tables(soup):
    tables = [table for table in soup.find_all("table") if is_launch_table(table)]
    if not tables:
        raise ValueError("No launch tables matching the expected schema were found.")
    return tables

## 5. Parse launch rows

A dedicated row parser separates extraction logic from iteration logic. This makes the code easier to test, maintain, and audit. Rows are accepted only when the row header begins with a numeric flight number and contains the expected number of data cells.

In [5]:
OUTPUT_COLUMNS = [
    "Flight No.",
    "Launch site",
    "Payload",
    "Payload mass",
    "Orbit",
    "Customer",
    "Launch outcome",
    "Version Booster",
    "Booster landing",
    "Date",
    "Time",
]


def parse_launch_row(row):
    """Parse one historical launch-table row into the project schema."""
    header = row.find("th")
    cells = row.find_all("td")

    if header is None or len(cells) < 9:
        return None

    flight_text = clean_text(header)
    match = re.match(r"^(\d+)", flight_text or "")
    if not match:
        return None

    date, time = parse_date_time(cells[0])

    return {
        "Flight No.": int(match.group(1)),
        "Launch site": first_link_text(cells[2]),
        "Payload": first_link_text(cells[3]),
        "Payload mass": parse_payload_mass(cells[4]),
        "Orbit": first_link_text(cells[5]),
        "Customer": clean_text(cells[6]),
        "Launch outcome": clean_text(cells[7]),
        "Version Booster": parse_booster_version(cells[1]),
        "Booster landing": parse_landing_status(cells[8]),
        "Date": date,
        "Time": time,
    }


def scrape_launch_records(soup):
    """Extract all valid historical launch rows into a dataframe."""
    records = []

    for table in find_launch_tables(soup):
        for row in table.find_all("tr"):
            record = parse_launch_row(row)
            if record is not None:
                records.append(record)

    if not records:
        raise ValueError("Launch tables were found, but no launch records were extracted.")

    return pd.DataFrame(records, columns=OUTPUT_COLUMNS)

## 6. Build the dataset

When live scraping is enabled, the notebook downloads and parses the fixed Wikipedia revision. In archived mode, it loads the historical CSV produced by the original successful scraper. Both modes populate the same `df` object and preserve the original 11-column schema.

In [6]:
if RUN_LIVE_SCRAPE:
    soup = fetch_wikipedia_snapshot()
    df = scrape_launch_records(soup)
    source_mode = "Live scrape of fixed Wikipedia revision"
else:
    if not ARCHIVED_OUTPUT.exists():
        raise FileNotFoundError(
            f"{ARCHIVED_OUTPUT} was not found. "
            "Place the archived CSV beside this notebook or set RUN_LIVE_SCRAPE = True."
        )

    df = pd.read_csv(ARCHIVED_OUTPUT)
    source_mode = "Archived output from original successful scrape"

# Preserve column order used by the original project.
df = df[OUTPUT_COLUMNS].copy()

print(f"Source mode: {source_mode}")
print(f"Dataset shape: {df.shape}")
df.head()

Source mode: Archived output from original successful scrape
Dataset shape: (228, 11)


,Flight No.,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Version Booster,Booster landing,Date,Time
0,1,CCAFS,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,F9 v1.07B0003.1,Failure,4 June 2010,18:45
1,1,CCAFS,Dragon,0,LEO,NASA,Success,F9 v1.07B0003.1,Failure,4 June 2010,18:45
2,2,CCAFS,Dragon,525 kg,LEO,NASA,Success,F9 v1.07B0004.1,No,8 December 2010,15:43
3,3,CCAFS,SpaceX CRS-1,"4,700 kg",LEO,NASA,Success,F9 v1.07B0005.1,No attempt,22 May 2012,07:44
4,4,CCAFS,SpaceX CRS-2,"4,877 kg",LEO,NASA,Success,F9 v1.07B0006.1,No,8 October 2012,00:35


## 7. Data-quality validation

The web-scraped dataset is checked before export. These checks are **intentionally lightweight** because this notebook's primary purpose is **acquisition**; substantive transformation and modeling occur later in the project.

The validation focuses on:
- schema consistency;
- duplicate rows;
- missingness;
- flight-number range;
- date coverage; and
- categorical coverage of key operational fields.

In [7]:
validation = pd.Series({
    "rows": len(df),
    "columns": df.shape[1],
    "duplicate_rows": int(df.duplicated().sum()),
    "min_flight_number": pd.to_numeric(df["Flight No."], errors="coerce").min(),
    "max_flight_number": pd.to_numeric(df["Flight No."], errors="coerce").max(),
    "unique_launch_sites": df["Launch site"].nunique(dropna=True),
    "unique_orbits": df["Orbit"].nunique(dropna=True),
}, name="value")

validation.to_frame()

,value
rows,228
columns,11
duplicate_rows,0
min_flight_number,1
max_flight_number,121
unique_launch_sites,5
unique_orbits,8


In [8]:
missingness = (
    df.isna()
      .sum()
      .rename("missing_count")
      .to_frame()
)

missingness["missing_pct"] = (
    100 * missingness["missing_count"] / len(df)
).round(2)

missingness

,missing_count,missing_pct
Flight No.,0,0.00
Launch site,1,0.44
Payload,1,0.44
Payload mass,1,0.44
Orbit,1,0.44
Customer,3,1.32
Launch outcome,2,0.88
Version Booster,0,0.00
Booster landing,2,0.88
Date,0,0.00


In [9]:
date_series = pd.to_datetime(df["Date"], errors="coerce", dayfirst=True)

date_coverage = pd.Series({
    "earliest_date": date_series.min(),
    "latest_date": date_series.max(),
    "unparsed_dates": int(date_series.isna().sum()),
}, name="value")

date_coverage.to_frame()

,value
earliest_date,2010-06-04 00:00:00
latest_date,2021-06-06 00:00:00
unparsed_dates,0


### Interpretation of the validation stage

The validation section is not intended to "clean away" all missing values. Some missing cells in the historical Wikipedia tables reflect incomplete or inapplicable source information rather than parser failure. The purpose here is to make those conditions visible before the dataset is used elsewhere.

A particularly important distinction is that `Payload mass` remains in the **historical text representation**. Numeric conversion can be performed later if a downstream analysis requires it; preserving the raw scraped representation at the acquisition stage maintains traceability to the source page.

## 8. Export the validated dataset

The validated dataframe is written to `spacex_web_scraped_portfolio.csv`, leaving the archived source file untouched. When `RUN_LIVE_SCRAPE = True`, the same export cell writes the newly scraped historical records.

In [10]:
df.to_csv(EXPORT_PATH, index=False)
print(f"Saved {len(df):,} records to: {EXPORT_PATH.resolve()}")

Saved 228 records to: /mnt/data/spacex_web_scraped_portfolio.csv


## 9. Key takeaways

This notebook demonstrates a complete semi-structured data-acquisition workflow:

- a fixed historical webpage is used to reduce source drift;
- request handling includes timeouts and HTTP-status validation;
- table discovery is based on expected schema rather than hard-coded table position;
- parsing responsibilities are separated into reusable functions;
- defensive extraction handles links, annotations, and inconsistent HTML content;
- the original 11-column output schema is preserved;
- data-quality checks make missingness and coverage transparent before export;
- an archived-output mode ensures that the portfolio remains inspectable and executable even when network access is unavailable.

### Methodological limitation

Web scraping is inherently dependent on external page availability and HTML structure. A fixed Wikipedia revision substantially improves reproducibility, but it cannot guarantee that the external host will remain permanently accessible. Preserving the resulting CSV alongside the notebook therefore provides an important reproducibility safeguard.

### Relationship to Notebook 01

Notebook 01 demonstrates **structured API acquisition and entity enrichment**. This notebook demonstrates **semi-structured HTML extraction and transformation**. Together, they show two complementary approaches to constructing analytical datasets from external operational data sources.

## Project attribution

This portfolio notebook is based on the web-scraping stage of the **IBM Data Science Professional Certificate SpaceX capstone**. The original project objective and historical source are retained, while the implementation has been refactored for clearer software structure, robustness, reproducibility, and portfolio presentation.